# 04 - Deploy the MAF agent with the Python SDK

Move the change-assessment agent from notebook 03 into **Foundry Hosted Agents**. It still discovers and loads the skill through the Toolbox published in notebook 02; hosting changes where Python runs, not how the skill is distributed.

Deploying a hosted agent from source requires application code, an entry point and runtime dependencies. **For this tutorial, these files have already been created for you** in `hosted_agent`. Review them below before packaging and uploading them.

**Flow:** review files -> create a ZIP in Python -> upload with the Foundry SDK -> wait for the hosted version -> invoke and verify. Foundry builds the runtime dependencies. This source-deployment path needs **no azd, Dockerfile, Docker installation or Azure Container Registry**.

The walkthrough reuses the existing Foundry project, model and published Toolbox. Deployment and invocation incur charges. It does not configure a scalability test.

## 1. Review the supplied files

Open these files in order to understand what each one does:

| File | What to review |
|---|---|
| [hosted_agent/agent.py](../hosted_agent/agent.py) | `create_agent` builds the same MAF agent as notebook 03. Find `MCPSkillsSource`, `SkillsProvider` and `context_providers`: instructions are loaded from Toolbox, not copied into the base prompt. `ToolboxAuth` obtains authentication for each HTTP request; `SkillLoadLogger` records successful instruction loads without logging their content. |
| [hosted_agent/main.py](../hosted_agent/main.py) | `ResponsesHostServer` exposes the agent through the official hosting adapter on port 8088. The MCP session stays open for the server's lifetime and closes with it. |
| [hosted_agent/requirements.txt](../hosted_agent/requirements.txt) | Runtime dependencies, including the preview hosting adapter. Foundry installs these during `remote_build`; Jupyter is not required in the hosted process. |

The notebook supplies deployment configuration through `HostedAgentDefinition` and `CodeConfiguration`, rather than a deployment YAML file. Python packages only the files above; credentials, virtual environments, notebooks and local artifacts are excluded.

The agent uses `DefaultAzureCredential`: Azure CLI can authenticate local testing; Foundry supplies the hosted agent identity at runtime. Personal CLI credentials are never uploaded. Only trusted skill-loading approval is disabled, as in notebook 03.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'requirements.txt').is_file():
    ROOT = ROOT.parent
SOURCE = ROOT / 'hosted_agent'
UPLOAD_FILES = ('agent.py', 'main.py', 'requirements.txt')
missing = [name for name in UPLOAD_FILES if not (SOURCE / name).is_file()]
if missing or not (ROOT / '.env.example').is_file():
    raise FileNotFoundError(f'Open the complete repository; missing runtime files: {missing}')
print('The supplied runtime files are present. Review their links above before continuing.')

## 2. Configure source deployment

Complete notebooks 01-03 first. Use the same Azure CLI sign-in and root environment file. There are no CLI subprocesses for packaging, upload, invocation or logs in this notebook.

- The deploying account needs **Foundry Project Manager** access. The hosted agent identity needs **Foundry User** access on the project to use the model and Toolbox.
- Complete the notebook-04 settings in [.env.example](../.env.example). Use a dedicated `HOSTED_AGENT_NAME`: deploying it again creates another agent version, independently of skill and Toolbox versions.
- Set `HOSTED_AGENT_RUNTIME` to a supported Foundry Python runtime, for example `python_3_13`. This is the hosted runtime, not the notebook kernel version.
- Set `PYTHON_PACKAGE_INDEX_URL` to an approved, credential-free HTTPS package feed. The ZIP's requirements file explicitly selects this feed. The remote builder must be able to reach it. A blocked or authenticated-only feed is a deployment prerequisite to resolve, not a reason to fall back silently to public PyPI.
- Confirm Hosted Agent availability and quota in the project's region. Model token limits and hosted-compute capacity are separate.

Keep the published Toolbox endpoint to follow its default configuration, or pin a Toolbox version for a repeatable deployment. Skills within a Toolbox configuration remain pinned. A running provider may cache skill metadata.

**Optional local review:** create a runtime environment using your configured approved feed, then press **F5** and select **Debug hosted change assessor**. The supplied VS Code configuration opens Agent Inspector; stop the debug task after testing.

```powershell
python -m venv hosted_agent\.venv
.\hosted_agent\.venv\Scripts\python.exe -m pip install -r hosted_agent\requirements.txt
```

Leave `DEPLOY` false while reviewing. Set it to true only when you intend to upload and invoke a billable hosted agent. Restart the kernel after changing the environment file.

In [ ]:
import hashlib
import os
import re
import time
from urllib.parse import urlsplit
from zipfile import ZIP_DEFLATED, ZipFile

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeConfiguration, HostedAgentDefinition, ProtocolVersionRecord, VersionRefIndicator
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(ROOT / '.env', override=False, interpolate=False)
DEPLOY = False

def required(name):
    value = os.environ.get(name, '').strip()
    if not value or '<' in value or '>' in value or '\n' in value or '\r' in value:
        raise ValueError(f'Populate {name} in the root environment file.')
    return value

PROJECT_ENDPOINT = required('FOUNDRY_PROJECT_ENDPOINT').rstrip('/')
MODEL = required('FOUNDRY_MODEL')
AGENT_NAME = required('HOSTED_AGENT_NAME')
RUNTIME = required('HOSTED_AGENT_RUNTIME')
FEED = required('PYTHON_PACKAGE_INDEX_URL')
endpoint = urlsplit(PROJECT_ENDPOINT)
if (endpoint.scheme != 'https' or not endpoint.hostname or not endpoint.hostname.endswith('.services.ai.azure.com')
    or endpoint.username or endpoint.password or endpoint.port not in (None, 443) or endpoint.query or endpoint.fragment
    or not re.fullmatch(r'/api/projects/[^/]+', endpoint.path)):
    raise ValueError('Use an HTTPS Foundry project endpoint.')
if not re.fullmatch(r'[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?', AGENT_NAME):
    raise ValueError('Use an alphanumeric/hyphen agent name, at most 63 characters, without leading/trailing hyphens.')
if not re.fullmatch(r'python_3_[0-9]+', RUNTIME):
    raise ValueError('Set HOSTED_AGENT_RUNTIME to a supported Foundry Python runtime identifier.')
feed = urlsplit(FEED)
if feed.scheme != 'https' or not feed.hostname or feed.username or feed.password or feed.query or feed.fragment or any(c.isspace() for c in FEED):
    raise ValueError('Use a credential-free HTTPS package feed URL without whitespace.')
TOOLBOX_NAME = required('FOUNDRY_TOOLBOX_NAME')
TOOLBOX_VERSION = os.environ.get('FOUNDRY_TOOLBOX_VERSION', '').strip()
if not re.fullmatch(r'[a-zA-Z0-9_-]+', TOOLBOX_NAME) or (TOOLBOX_VERSION and not re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9_.-]*', TOOLBOX_VERSION)):
    raise ValueError('Invalid Toolbox name or version.')
suffix = f'/versions/{TOOLBOX_VERSION}' if TOOLBOX_VERSION else ''
TOOLBOX_ENDPOINT = f'{PROJECT_ENDPOINT}/toolboxes/{TOOLBOX_NAME}{suffix}/mcp?api-version=v1'
subscription = os.environ.get('AZURE_SUBSCRIPTION_ID', '').strip()
tenant = os.environ.get('AZURE_TENANT_ID', '').strip()
CREDENTIAL_OPTIONS = {'subscription': subscription} if subscription else ({'tenant_id': tenant} if tenant else {})
print('Agent:', AGENT_NAME, '| runtime:', RUNTIME)
print('Toolbox selection:', TOOLBOX_VERSION or 'published default')
print('Deployment enabled:', DEPLOY)

## 3. Package the source in Python

Create a ZIP containing only `agent.py`, `main.py` and `requirements.txt` at its root. The requirements copy inside the ZIP selects the approved feed; the supplied source file is not modified. No local skill body is included: the running agent must retrieve its instructions from Toolbox.

Inspect the printed ZIP entries, size and digest. The SDK accepts a ZIP file stream up to 250 MB and uses its SHA-256 digest for integrity. This cell creates a local, ignored artifact only; it does not deploy anything.

In [ ]:
ARTIFACTS = ROOT / '.demo-artifacts' / 'sdk-source'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ARTIFACTS / 'hosted-agent.zip'
requirements = (SOURCE / 'requirements.txt').read_text(encoding='utf-8')
with ZipFile(ZIP_PATH, 'w', compression=ZIP_DEFLATED) as package:
    for name in ('agent.py', 'main.py'):
        package.write(SOURCE / name, arcname=name)
    package.writestr('requirements.txt', f'--index-url {FEED}\n{requirements}')
with ZipFile(ZIP_PATH) as package:
    if sorted(package.namelist()) != sorted(UPLOAD_FILES):
        raise AssertionError('The ZIP must contain only the three reviewed runtime files.')
    print('ZIP entries:', package.namelist())
if ZIP_PATH.stat().st_size > 250 * 1024 * 1024:
    raise ValueError('The source ZIP exceeds the SDK upload limit.')
ZIP_SHA256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print('ZIP bytes:', ZIP_PATH.stat().st_size)
print('SHA-256:', ZIP_SHA256)

## 4. Upload a hosted agent version with the SDK

`HostedAgentDefinition` declares compute, the Responses protocol and runtime settings. `CodeConfiguration` specifies the Python runtime, `python main.py` entry point and **remote_build** dependency resolution. Foundry prepares the runtime; you do not build or push a container yourself.

The deployed environment receives the model deployment name and Toolbox URL. Foundry injects the project endpoint and agent identity; no notebook-user credentials are passed. The managed skill stays outside this package.

Run the upload cell **once per intended agent version**. Responses protocol **2.0.0** lets Foundry supply the trusted user and call identifiers required by the hosting adapter. A successful upload is not yet proof that the version is ready or that its application works. Note the returned version; if a later check fails, inspect that exact version rather than uploading again.

In [ ]:
if not DEPLOY:
    raise RuntimeError('Review the package and prerequisites, then set DEPLOY = True in configuration.')
definition = HostedAgentDefinition(
    cpu='1', memory='2Gi',
    code_configuration=CodeConfiguration(
        runtime=RUNTIME,
        entry_point=['python', 'main.py'],
        dependency_resolution='remote_build',
    ),
    protocol_versions=[ProtocolVersionRecord(protocol='responses', version='2.0.0')],
    environment_variables={
        'AZURE_AI_MODEL_DEPLOYMENT_NAME': MODEL,
        'TOOLBOX_ENDPOINT': TOOLBOX_ENDPOINT,
    },
)
with AzureCliCredential(**CREDENTIAL_OPTIONS, process_timeout=60) as credential:
    with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
        with ZIP_PATH.open('rb') as code:
            uploaded = project.agents.create_version_from_code(
                AGENT_NAME, definition=definition, code=code, code_zip_sha256=ZIP_SHA256,
                description='MAF change assessment using skills loaded from Foundry Toolbox.',
            )
AGENT_VERSION = uploaded.version
print('Uploaded agent:', AGENT_NAME, '| version:', AGENT_VERSION, '| status:', uploaded.status)

## 5. Check readiness and the hosted identity

Poll the **exact uploaded version**, not the newest version of the agent. Stop explicitly if it fails or does not become active within the bounded wait. An active version still needs a successful invocation to validate runtime startup, model access and Toolbox access.

The identity printed below belongs to the hosted agent, not the notebook user. Confirm that it has **Foundry User** access on the project; grant that role in the project's access controls if it has not already been assigned. If deployment fails, inspect the version's build diagnostics in Foundry. In particular, verify access to the configured package feed; do not silently substitute another feed.

In [ ]:
deadline = time.monotonic() + 900
with AzureCliCredential(**CREDENTIAL_OPTIONS, process_timeout=60) as credential:
    with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
        deployed = project.agents.get_version(AGENT_NAME, AGENT_VERSION)
        while deployed.status == 'creating':
            if time.monotonic() >= deadline:
                raise TimeoutError(f'Agent {AGENT_NAME}/{AGENT_VERSION} is still creating. Inspect this version before retrying readiness; do not re-upload.')
            print('Status:', deployed.status)
            time.sleep(10)
            deployed = project.agents.get_version(AGENT_NAME, AGENT_VERSION)
if deployed.status != 'active':
    raise RuntimeError(f'Hosted version did not become active: {deployed.as_dict()}')
print('Active hosted version:', deployed.version)
if deployed.instance_identity is None:
    raise RuntimeError('No hosted agent identity was returned; inspect this version before invoking it.')
print('Hosted identity principal ID:', deployed.instance_identity.principal_id)

### Check the deployment in Foundry

After the readiness cell completes, open your project in the [Foundry portal](https://ai.azure.com/) and select **Agents**. You should see `skills-change-assessor`, its deployed version, type **Hosted**, and status **Running**, as illustrated below. Refresh the list if needed. Your version number may differ from the example.

![Foundry Agents list showing skills-change-assessor version 2 with status Running and type Hosted](images/foundry-hosted-agent.png)

This confirms that the deployment is visible in Foundry. It does not prove successful invocation, skill loading, or scalability. The following cells verify the remote assessment and actual skill-load evidence.

## 6. Invoke the hosted version

After checking the hosted identity's access, create a session pinned to the uploaded version with `VersionRefIndicator`, then call the agent-specific Responses endpoint. Hosted agents cannot be invoked through the project's shared Responses endpoint. This is a remote request; the notebook does not instantiate a local MAF Agent. It uses the same task as notebook 03, without supplying any skill body.

**Expected result:** an assessment with `clarification-needed`, ten ordered report sections and the three supplied evidence citations. A cold session may take longer to start. Permission failures, quota errors and timeouts remain explicit; they are not replaced with sample responses.

In [ ]:
if not DEPLOY:
    raise RuntimeError('Enable deployment explicitly before invoking the remote agent.')
TASK = '''Assess this proposed change before implementation planning.

[CHANGE-01] Rename the required JSON response field `order_id` to `id` in
GET /orders/{order_id}. The requested outcome is consistent field naming.
[CONTRACT-01] The current response contains `order_id` and `status`.
[CONSTRAINT-01] No decision has been made about retaining the old field,
API versioning, or a deprecation period. Consumer inventory is unavailable.

Use only this supplied context; repository access is not available.
Do not implement the change or claim that tests have run.
'''
with AzureCliCredential(**CREDENTIAL_OPTIONS, process_timeout=60) as credential:
    with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
        session = project.agents.create_session(
            AGENT_NAME, version_indicator=VersionRefIndicator(agent_version=AGENT_VERSION),
        )
        SESSION_ID = session.agent_session_id
        if not isinstance(SESSION_ID, str) or not SESSION_ID:
            raise RuntimeError('Session creation did not return an agent_session_id.')
        print('Created session:', SESSION_ID, '| pinned version:', AGENT_VERSION, flush=True)
        with project.get_openai_client(agent_name=AGENT_NAME, timeout=180, max_retries=0) as client:
            response = client.responses.create(input=TASK, extra_body={'agent_session_id': SESSION_ID})
if response.status != 'completed' or not response.output_text.strip():
    raise RuntimeError(f'Hosted assessment did not complete: {response.status}; error={response.error}')
if (response.model_extra or {}).get('agent_session_id') != SESSION_ID:
    raise RuntimeError('The response did not confirm the requested agent_session_id; inspect the hosted request.')
print('Hosted version:', AGENT_VERSION, '| session:', SESSION_ID, '| response:', response.id)
print(response.output_text)

## 7. Check the report and actual instruction loading

The report checks below cover visible effects of the skill, not complete assessment quality. Passing them alone does not prove that the skill was retrieved.

**Expected report:** `clarification-needed` in the Verdict, ten ordered assessment sections, and citations to `CHANGE-01`, `CONTRACT-01` and `CONSTRAINT-01`.

In [ ]:
sections = (
    'Verdict', 'Requested change', 'Evidence', 'Assumptions and unknowns',
    'Affected components', 'Backward compatibility', 'Risks and dependencies',
    'Verification strategy', 'Clarifying questions', 'Recommended next action',
)
report = response.output_text
labels = re.findall(
    r'(?im)^[ ]{0,3}(?:#{1,6}[ \t]+|\d+[.)][ \t]+)?(?:\*\*)?(' + '|'.join(re.escape(s) for s in sections) + r')(?:\*\*)?[ \t]*(?=:|$)',
    report,
)
found = [label.lower() for label in labels]
if found != [section.lower() for section in sections]:
    raise AssertionError(f'Expected ten ordered report sections; found {found}')
verdict = re.split(r'(?i)Requested change', report, maxsplit=1)[0]
if not re.search(r'\bclarification-needed\b', verdict):
    raise AssertionError('Expected clarification-needed for the unresolved compatibility decision.')
for citation in ('CHANGE-01', 'CONTRACT-01', 'CONSTRAINT-01'):
    if citation not in report:
        raise AssertionError(f'Missing evidence citation: {citation}')
print('Hosted report checks passed. Verify the runtime load event below.')

### Read this session's logs through the SDK

`get_session_log_stream` streams console logs as SSE bytes. Its `cls` response hook exposes the HTTP stream so we can close it as soon as the evidence is found. Treat log payloads as opaque text; their internal JSON schema is not contractual.

**Expected runtime evidence:** `change_assessor skill_loaded instruction_characters=<N>`. `SkillLoadLogger` writes this only after a real `load_skill` call returns nonempty instructions. It is not the model claiming to have used the skill. The stream is scoped to the exact agent version and returned session ID.

The bounded check fails if the event is missing. If console delivery is delayed, inspect the session in Foundry and rerun only this log check, not upload or invocation.

In [ ]:
import codecs

def log_http_response(pipeline_response, deserialized, response_headers):
    return pipeline_response.http_response

pattern = re.compile(r'change_assessor\s+skill_loaded instruction_characters=[1-9][0-9]*')
verified = False
decoder = codecs.getincrementaldecoder('utf-8')()
buffer = ''
deadline = time.monotonic() + 60
with AzureCliCredential(**CREDENTIAL_OPTIONS, process_timeout=60) as credential:
    with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
        with project.agents.get_session_log_stream(
            AGENT_NAME, AGENT_VERSION, SESSION_ID, cls=log_http_response, read_timeout=60,
        ) as stream:
            for chunk in stream.iter_bytes():
                buffer += decoder.decode(chunk)
                if pattern.search(buffer):
                    verified = True
                    break
                if len(buffer) > 1024 * 1024:
                    raise RuntimeError('Log evidence was not found within the first MiB of session logs.')
                if time.monotonic() >= deadline:
                    break
if not verified:
    raise AssertionError('No successful skill-load event found for this session. Inspect its logs before retrying this check.')
print('Verified a successful load_skill execution in the hosted session.')

### Inspect skill loading in the Playground

You can also test the deployed agent interactively: open your project in the [Foundry portal](https://ai.azure.com/), select **Agents** > **skills-change-assessor** > **Playground**, and submit the same change request. Open **Log stream** and filter for `skill` to find `Function load_skill succeeded.` alongside the assessment, as illustrated below.

![Foundry Playground showing the assessment and a filtered log entry reporting Function load_skill succeeded](images/foundry-hosted-skill-load.png)

This framework log shows a successful `load_skill` invocation, rather than the model merely claiming to use a skill. The notebook's check above is stricter: it requires the `change_assessor skill_loaded instruction_characters=<N>` event, written only after nonempty instructions are returned.

The Playground creates its own session; this screenshot illustrates a separate interactive test, not the SDK session checked above. Session IDs, timestamps and version numbers will differ.

## 8. Stop the test session

Run this cleanup cell after collecting evidence, or after a later check fails if a session ID was returned. It stops this test session's compute without deleting its filesystem. The deployed agent version remains available for a new session and a future scalability notebook.

Agent versions, skill versions and Toolbox versions are independent. Retain the tested agent version for later measurements of independent sessions, latency, throughput, throttling and failures. Deployment alone does not demonstrate scalability, crash recovery or production reliability.

Review retained session files and remove only demo resources you no longer need. Do not delete the shared project, model or resource group. Evaluation jobs and load tests are separate, explicitly approved steps.

In [ ]:
with AzureCliCredential(**CREDENTIAL_OPTIONS, process_timeout=60) as credential:
    with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
        project.agents.stop_session(AGENT_NAME, SESSION_ID)
print('Stopped the test session; the hosted agent version is retained.')